In [1]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"   

In [2]:
from datasets import load_dataset

imdb= load_dataset("stanfordnlp/imdb")
print(imdb)
train= imdb["train"]
test= imdb["test"]

print(train.features["label"].names)
print("Label:", train[0]["label"])
print(train[0]["text"][:400])

Using the latest cached version of the dataset since stanfordnlp/imdb couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'plain_text' at C:\Users\Puruj\.cache\huggingface\datasets\stanfordnlp___imdb\plain_text\0.0.0\e6281661ce1c48d982bc483cf8a173c1bbeb5d31 (last modified on Tue Oct 13 18:16:50 2026).


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
['neg', 'pos']
Label: 0
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student name


In [3]:
from collections import Counter
import numpy as np

print("Label Counts", Counter(train["label"]))

print("First 5: ", train["label"][:5])
print("Last 5: ", train["label"][-5:])

lengths = np.array([len(text.split()) for text in train["text"]])
print("Words per review: min ", lengths.min(), "|median " , int(np.median(lengths)), "|mean ", round(lengths.mean()), "|max ", lengths.max())

print("Reviews with <br /> :" , sum("<br />" in text for text in train["text"]))

Label Counts Counter({0: 12500, 1: 12500})
First 5:  [0, 0, 0, 0, 0]
Last 5:  [1, 1, 1, 1, 1]
Words per review: min  10 |median  174 |mean  234 |max  2470
Reviews with <br /> : 14665


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

tiny= ["this movie was great fun",
      "this movie was awful",
      "great acting but awful plot"]

vec = TfidfVectorizer()
tiny_numbers= vec.fit_transform(tiny)

print(vec.get_feature_names_out())
print(tiny_numbers.toarray().round(2))

['acting' 'awful' 'but' 'fun' 'great' 'movie' 'plot' 'this' 'was']
[[0.   0.   0.   0.55 0.42 0.42 0.   0.42 0.42]
 [0.   0.5  0.   0.   0.   0.5  0.   0.5  0.5 ]
 [0.49 0.37 0.49 0.   0.37 0.   0.49 0.   0.  ]]


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LogisticRegression(max_iter=1000)),
])

tfidf_model.fit(train["text"], train["label"])
test_preds= tfidf_model.predict(test["text"])

print("Vocab size: ", len(tfidf_model.named_steps["tfidf"].vocabulary_))
print("Test_accuracy: ", round(100 * accuracy_score(test["label"], test_preds), 2), "&")

Vocab size:  74849
Test_accuracy:  88.29 &


In [6]:
import numpy as np

words= tfidf_model.named_steps["tfidf"].get_feature_names_out()
weights = tfidf_model.named_steps["model"].coef_[0]
order= np.argsort(weights)

print("Most NEGATIVE words:", [(words[i], round(weights[i], 1)) for i in order[:5]])
print("Most POSITIVE words:",[(words[i], round(weights[i], 1)) for i in order[::-1][:5]])

my_reviews= [
     "An absolute masterpiece. I loved every minute of it.",
    "Two hours I will never get back. Boring and badly acted.",
    "I expected to hate it, but honestly it was not bad at all.",
]

probs= tfidf_model.predict_proba(my_reviews)
for review ,p in zip(my_reviews, probs):  
   print(round(p[1] * 100), "% positive →", review)

Most NEGATIVE words: [('worst', np.float64(-9.4)), ('bad', np.float64(-7.7)), ('awful', np.float64(-6.6)), ('waste', np.float64(-6.4)), ('boring', np.float64(-6.2))]
Most POSITIVE words: [('great', np.float64(7.4)), ('excellent', np.float64(6.4)), ('best', np.float64(5.1)), ('perfect', np.float64(5.0)), ('wonderful', np.float64(4.8))]
91 % positive → An absolute masterpiece. I loved every minute of it.
4 % positive → Two hours I will never get back. Boring and badly acted.
4 % positive → I expected to hate it, but honestly it was not bad at all.


In [7]:
import re
from collections import Counter

def tokenize(text):
    text= text.lower().replace("<br />", " ")
    return re.findall(r"['a-z0-9']+", text)

print(tokenize("I LOVED it!<br /><br />Not bad, 10/10."))

word_counts= Counter()
for text in train["text"]:
    word_counts.update(tokenize(text))
print("Distinct words:", len(word_counts))
print("Most common:", word_counts.most_common(5))

vocab= {"<pad>":0 , "<unk>":1}
for word, count in word_counts.most_common(19998):
    vocab[word] = len(vocab)
print("Vocab size:", len(vocab), "| the=", vocab["the"], "| bad=", vocab["bad"], "| great=", vocab["great"])

def encode(text):
    return [vocab.get(word,1) for word in tokenize(text)]
print(encode("Not bad at all, a masterpiece of zxqv!"))

['i', 'loved', 'it', 'not', 'bad', '10', '10']
Distinct words: 87171
Most common: [('the', 336195), ('and', 164136), ('a', 163093), ('of', 145850), ('to', 135713)]
Vocab size: 20000 | the= 2 | bad= 76 | great= 85
[21, 76, 30, 29, 4, 990, 5, 1]


In [8]:
import torch
from torch.utils.data import DataLoader, TensorDataset

MAX_LEN=256
def pad(ids):
    ids= ids[:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

print(pad([21, 76, 30, 29])[:8], "...length", len(pad([21, 76, 30, 29])))

X_train = torch.tensor([pad(encode(t)) for t in train["text"]])
y_train = torch.tensor(train["label"])
X_test = torch.tensor([pad(encode(t)) for t in test["text"]])
y_test = torch.tensor(test["label"])
print(X_train.shape, y_train.shape, X_test.shape)

text_train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size= 64,  shuffle= True)
text_test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64)

xb,yb= next(iter(text_train_loader))
print("One batch:", xb.shape, yb.shape, "| batches per epoch:" , len(text_train_loader))

[21, 76, 30, 29, 0, 0, 0, 0] ...length 256
torch.Size([25000, 256]) torch.Size([25000]) torch.Size([25000, 256])
One batch: torch.Size([64, 256]) torch.Size([64]) | batches per epoch: 391


In [9]:
from torch import nn

emb = nn.Embedding(25000 ,  64,  padding_idx=0)

ids= torch.tensor([encode("not bad at all")])
vector= emb(ids)

print("IDs:" , ids, ids.shape)
print("After embedding:", vector.shape)
print("First 5 numbers of 'not':" , vector[0, 0, :5].detach())

IDs: tensor([[21, 76, 30, 29]]) torch.Size([1, 4])
After embedding: torch.Size([1, 4, 64])
First 5 numbers of 'not': tensor([-0.0748,  0.3799, -1.2530, -0.5169, -0.8440])


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

class TextCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(20000, 64, padding_idx= 0)
        self.conv= nn.Conv1d(64, 128, kernel_size=3, padding=1)
        self.relu= nn.ReLU()
        self.drop= nn.Dropout(0.5)
        self.fc= nn.Linear(128, 2)

    def forward(self, x):
        
        x= self.embed(x)
        x= x.permute(0, 2, 1)
        x= self.relu(self.conv(x))
        x= x.max(dim=2).values
        return self.fc(self.drop(x))

text_model= TextCNN().to(device)
print("Weights to learn: ", sum(p.numel() for p in text_model.parameters()))
out= text_model(xb.to(device))
print("Batch in:", xb.shape, "--> out:", out.shape)

Weights to learn:  1304962
Batch in: torch.Size([64, 256]) --> out: torch.Size([64, 2])


In [11]:
loss_fn = nn.CrossEntropyLoss()
optimizer= torch.optim.Adam(text_model.parameters(), lr= 0.001 )

def accuracy(model, loader):
    model.eval()
    correct, total =0, 0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device), y.to(device)
            correct += (model(x).argmax(dim=1)==y).sum().item()
            total += y.size(0)
        return 100*correct/ total

for epoch in range(6):
    text_model.train()
    for x,y in text_train_loader:
        x,y= x.to(device), y.to(device)
        loss= loss_fn(text_model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print("epoch", epoch+1,
          "train", round(accuracy(text_model, text_train_loader), 2),
          "test", round(accuracy(text_model, text_test_loader), 2),
    )
my_reviews = ["An absolute masterpiece. I loved every minute of it.",
              "Two hours I will never get back. Boring and badly acted.",
              "I expected to hate it, but honestly it was not bad at all."]
text_model.eval()
with torch.no_grad():
    ids= torch.tensor([pad(encode(r)) for r in my_reviews]).to(device)
    probs= torch.softmax(text_model(ids), dim=1)[:,1]
for r,p in zip(my_reviews, probs):
    print(round(p.item()*100), "% positive -->", r)

epoch 1 train 77.42 test 76.52
epoch 2 train 81.45 test 79.77
epoch 3 train 82.19 test 79.73
epoch 4 train 87.0 test 83.34
epoch 5 train 89.3 test 84.29
epoch 6 train 90.91 test 84.85
98 % positive --> An absolute masterpiece. I loved every minute of it.
2 % positive --> Two hours I will never get back. Boring and badly acted.
12 % positive --> I expected to hate it, but honestly it was not bad at all.


In [12]:
from transformers import AutoTokenizer

bert_tok= AutoTokenizer.from_pretrained("distilbert-base-uncased")
print(bert_tok.tokenize("I expected to hate it , but honestly it was not bad at all. Unbelievably good"))
enc= bert_tok(["not bad at all", "An absolute masterpiece, unbelievably good!"],
             padding= True, truncation= True, max_length=256, return_tensors= "pt")
print(enc["input_ids"])
print(enc["attention_mask"])
print(bert_tok.convert_ids_to_tokens(enc["input_ids"][0]))

['i', 'expected', 'to', 'hate', 'it', ',', 'but', 'honestly', 'it', 'was', 'not', 'bad', 'at', 'all', '.', 'un', '##bel', '##ie', '##va', '##bly', 'good']
tensor([[  101,  2025,  2919,  2012,  2035,   102,     0,     0,     0,     0,
             0,     0,     0],
        [  101,  2019,  7619, 17743,  1010,  4895,  8671,  2666,  3567,  6321,
          2204,   999,   102]])
tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])
['[CLS]', 'not', 'bad', 'at', 'all', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']


In [13]:
from torch.utils.data import DataLoader

def bert_tokenize(batch):
    return bert_tok(batch["text"], truncation= True, max_length= 256, padding="max_length")
bert_train= train.map(bert_tokenize, batched= True)
bert_test= test.map(bert_tokenize, batched= True)

cols= ["input_ids", "attention_mask", "label"]
bert_train.set_format("torch", columns= cols)
bert_test.set_format("torch", columns=cols)

bert_train_loader= DataLoader(bert_train, batch_size=16, shuffle= True)
bert_test_loader= DataLoader(bert_test, batch_size=64)

b= next(iter(bert_train_loader))
print(b.keys())
print(b["input_ids"].shape, b["attention_mask"].shape, b["label"].shape)
print("Batches per epoch:", len(bert_train_loader))

dict_keys(['label', 'input_ids', 'attention_mask'])
torch.Size([16, 256]) torch.Size([16, 256]) torch.Size([16])
Batches per epoch: 1563


In [14]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
device= "cuda"
bert_model= AutoModelForSequenceClassification.from_pretrained(
   "distilbert-base-uncased", num_labels=2
).to(device)
print("Parameters:", sum(p.numel() for p in bert_model.parameters()))

opt = torch.optim.AdamW(bert_model.parameters(), lr= 3e-5)
total_steps= len(bert_train_loader)
sched= get_linear_schedule_with_warmup(
    opt, num_warmup_steps= int(0.1*total_steps), num_training_steps= total_steps
)
e= bert_tok(["not bad at all", "terrible"], padding= True, return_tensors= "pt").to(device)
out= bert_model(input_ids= e["input_ids"], attention_mask= e["attention_mask"],
               labels=torch.tensor([1,0]).to(device))
print(out.loss)
print(out.logits)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Parameters: 66955010
tensor(0.7053, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor([[-0.0874,  0.1634],
        [-0.1144,  0.1515]], device='cuda:0', grad_fn=<AddmmBackward0>)


In [15]:
bert_model.train()
step=0 
for b in bert_train_loader:
    ids= b["input_ids"].to(device)
    mask= b["attention_mask"].to(device)
    y= b["label"].to(device)
   
    out= bert_model(input_ids=ids, attention_mask= mask , labels =y) 

    opt.zero_grad()
    out.loss.backward()
    opt.step()
    sched.step()

    step +=1
    if step % 200==0:
        print("step", step, "of", len(bert_train_loader), "loss", round(out.loss.item(), 3))
print("Training Done")

step 200 of 1563 loss 0.439
step 400 of 1563 loss 0.255
step 600 of 1563 loss 0.227
step 800 of 1563 loss 0.363
step 1000 of 1563 loss 0.246
step 1200 of 1563 loss 0.411
step 1400 of 1563 loss 0.078
Training Done


In [ ]:
bert_model.eval()
correct, total= 0,0
with torch.no_grad():
    for b in bert_test_loader:
        ids= b["input_ids"].to(device)
        mask= b["attention_mask"].to(device)
        y=b["label"].to(device)
        logits= bert_model(input_ids= ids, attention_mask=mask).logits
        correct += (logits.argmax(dim=1)==y).sum().item()
        total += y.size(0)
print("Distillbert test accuracy", round(100*correct/total, 2), "%")

In [ ]:
tricky = ["An absolute masterpiece. I loved every minute of it.",
          "Two hours I will never get back. Boring and badly acted.",
          "I expected to hate it, but honestly it was not bad at all.",
          "Not good. Not good at all.",
          "The acting was terrible and the plot was a mess, but somehow I loved it.",
          "I wanted to love this movie, but it was a huge disappointment."]

tfidf_probs = tfidf_model.predict_proba(tricky)[:, 1]

e = bert_tok(tricky, truncation= True, max_length=256, padding= True, return_tensors="pt").to(device)
with torch.no_grad():
    logits= bert_model(input_ids= e["input_ids"], attention_mask= e["attention_mask"]).logits
    bert_probs= torch.softmax(logits, dim=1)[:,1]
print("TF-IDF  BERT  review")
for r, t , p in zip(tricky, tfidf_probs, bert_probs):
    print(round(t*100), "%  ", round(p.item()*100), "%  ",r)

In [ ]:
bert_model.save_pretrained("models/distilbert-imdb")
bert_tok.save_pretrained("models/distilbert-imdb")
print("Saved")